[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/30.%20The%20Yard%20Pre-Marshalling%20for%20Exports%20Problem/P30-Tier-5_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 30. The Yard Pre-Marshalling for Exports Problem
## Tier 5: Integrated Digital Twin

### Goal
Elevate pre-marshalling from isolated optimization to comprehensive ecosystem simulation, creating a real-time virtual replica of the entire container terminal that enables predictive analysis, scenario planning, and system-wide optimization across interconnected operations.

### Key Assumptions
- Multi-model integration with federated simulation architecture
- Real-time synchronization between physical and virtual systems
- Event-driven interface for subsystem communication
- Predictive analytics with multiple time horizons

### Approach (Step-by-Step)
1. **Digital Twin Architecture**: Create integrated system-of-systems simulation framework
2. **Multi-Model Integration**: Connect pre-marshalling with vessel schedules, equipment, and logistics
3. **Predictive Analytics**: Implement multi-horizon forecasting (short, medium, long-term)
4. **Real-Time Adaptation**: Handle dynamic constraints and disruption scenarios
5. **What-If Analysis**: Enable comprehensive scenario testing and optimization
6. **Performance Monitoring**: Track system-wide KPIs and resilience metrics

### What to Look for in the Results
- System-wide performance improvements vs isolated optimization
- Predictive accuracy and forecast reliability
- Resilience to disruptions and unexpected events
- Cross-functional optimization benefits

### Concrete Example
Terminal: 12 export bays, 144 total stacks, 4 yard cranes
- Container volume: 2,400 TEU requiring pre-marshalling
- Vessel schedule: 3 ships departing within 18 hours
- External factors: 15% chance of rain (reduces crane speed by 20%)

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional, Any
import random
from collections import defaultdict, deque
import time
from copy import deepcopy
from datetime import datetime, timedelta
import itertools

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
@dataclass
class Container:
    """Container with comprehensive information"""
    id: str
    priority: int
    current_stack: int
    current_tier: int
    destination_vessel: str
    departure_time: float  # Hours from simulation start
    weight: float = 22.0  # TEU weight in tons
    type: str = "Standard"  # Container type

@dataclass
class VesselSchedule:
    """Vessel schedule information"""
    vessel_id: str
    arrival_time: float
    departure_time: float
    capacity: int
    assigned_berth: int
    container_requirements: List[str] = field(default_factory=list)

@dataclass
class EquipmentStatus:
    """Equipment availability and status"""
    equipment_id: str
    equipment_type: str  # 'YardCrane', 'QuayCrane', 'AGV'
    location: Tuple[int, int]  # (stack, tier) or berth
    status: str  # 'Available', 'Busy', 'Maintenance', 'Breakdown'
    speed: float = 1.0  # Relative speed factor
    fuel_consumption: float = 1.0

@dataclass
class WeatherCondition:
    """Weather impact on operations"""
    timestamp: float
    condition: str  # 'Clear', 'Rain', 'Wind', 'Fog'
    impact_factor: float  # Speed reduction factor
    duration: float  # Hours

@dataclass
class DigitalTwinEvent:
    """Event for digital twin synchronization"""
    timestamp: float
    event_type: str
    source_system: str
    data: Dict[str, Any]
    priority: int = 1

In [ ]:
class PreMarshallingModel:
    """Pre-marshalling optimization model"""
    
    def __init__(self, stacks: int, tiers: int, containers: List[Container]):
        self.stacks = stacks
        self.tiers = tiers
        self.containers = containers
        self.moves_executed = []
        self.total_moves = 0
        
    def count_violations(self) -> int:
        """Count priority violations"""
        violations = 0
        grid = [[None for _ in range(self.tiers)] for _ in range(self.stacks)]
        
        for container in self.containers:
            if container.current_tier < self.tiers:
                grid[container.current_stack][container.current_tier] = container.priority
        
        for stack in range(self.stacks):
            for tier in range(self.tiers - 1):
                lower = grid[stack][tier]
                upper = grid[stack][tier + 1]
                if lower is not None and upper is not None and lower > upper:
                    violations += 1
        
        return violations
    
    def optimize_marshalling(self, time_horizon: float) -> Dict[str, Any]:
        """Optimize pre-marshalling for given time horizon"""
        # Simplified optimization - prioritize containers with earlier departure
        urgent_containers = [c for c in self.containers if c.departure_time <= time_horizon]
        urgent_containers.sort(key=lambda x: x.departure_time)
        
        moves_needed = 0
        for container in urgent_containers:
            # Check if container is blocked
            if self.is_blocked(container):
                moves_needed += 1
        
        return {
            'moves_required': moves_needed,
            'urgent_containers': len(urgent_containers),
            'optimization_score': moves_needed / len(urgent_containers) if urgent_containers else 0
        }
    
    def is_blocked(self, container: Container) -> bool:
        """Check if container is blocked by higher priority containers"""
        stack_containers = [c for c in self.containers if c.current_stack == container.current_stack]
        stack_containers.sort(key=lambda x: x.current_tier)
        
        container_index = next(i for i, c in enumerate(stack_containers) if c.id == container.id)
        
        # Check if any container above has higher priority (lower number)
        for i in range(container_index + 1, len(stack_containers)):
            if stack_containers[i].priority < container.priority:
                return True
        
        return False

class VesselScheduleModel:
    """Vessel scheduling and berth allocation model"""
    
    def __init__(self, vessels: List[VesselSchedule], berths: int):
        self.vessels = vessels
        self.berths = berths
        self.berth_occupancy = defaultdict(list)
        
    def update_schedule(self, current_time: float, events: List[DigitalTwinEvent]):
        """Update vessel schedule based on events"""
        for event in events:
            if event.event_type == 'vessel_delay':
                vessel_id = event.data['vessel_id']
                delay = event.data['delay_hours']
                
                for vessel in self.vessels:
                    if vessel.vessel_id == vessel_id:
                        vessel.arrival_time += delay
                        vessel.departure_time += delay
                        break
        
        self.update_berth_occupancy(current_time)
    
    def update_berth_occupancy(self, current_time: float):
        """Update berth occupancy status"""
        self.berth_occupancy.clear()
        
        for vessel in self.vessels:
            if vessel.arrival_time <= current_time <= vessel.departure_time:
                self.berth_occupancy[vessel.assigned_berth].append(vessel.vessel_id)
    
    def get_loading_efficiency(self, current_time: float) -> float:
        """Calculate current loading efficiency"""
        active_vessels = sum(1 for v in self.vessels 
                           if v.arrival_time <= current_time <= v.departure_time)
        
        if active_vessels == 0:
            return 1.0
        
        # Efficiency based on berth utilization and vessel capacity
        total_capacity = sum(v.capacity for v in self.vessels 
                           if v.arrival_time <= current_time <= v.departure_time)
        
        return min(1.0, active_vessels / self.berths * total_capacity / (active_vessels * 100))

class EquipmentModel:
    """Equipment availability and performance model"""
    
    def __init__(self, equipment: List[EquipmentStatus]):
        self.equipment = equipment
        self.maintenance_schedule = defaultdict(list)
        self.breakdown_probability = 0.05  # 5% chance per hour
        
    def update_equipment_status(self, current_time: float, weather: WeatherCondition):
        """Update equipment status based on time and conditions"""
        # Apply weather effects
        if weather.condition == 'Rain':
            for equip in self.equipment:
                if equip.equipment_type in ['YardCrane', 'QuayCrane']:
                    equip.speed = 0.8  # 20% reduction in rain
        else:
            for equip in self.equipment:
                equip.speed = 1.0  # Reset to normal
        
        # Random breakdowns
        for equip in self.equipment:
            if equip.status == 'Available' and random.random() < self.breakdown_probability / 60:  # Per minute
                equip.status = 'Breakdown'
                equip.speed = 0.0
                
                # Schedule repair
                repair_time = current_time + random.uniform(1, 4)  # 1-4 hours repair
                self.maintenance_schedule[repair_time].append(equip.equipment_id)
        
        # Check for completed maintenance
        if current_time in self.maintenance_schedule:
            for equip_id in self.maintenance_schedule[current_time]:
                for equip in self.equipment:
                    if equip.equipment_id == equip_id:
                        equip.status = 'Available'
                        equip.speed = 1.0
                        break
        
        del self.maintenance_schedule[current_time]  # Remove processed maintenance
    
    def get_available_equipment(self, equipment_type: str) -> List[EquipmentStatus]:
        """Get available equipment of specific type"""
        return [e for e in self.equipment 
                if e.equipment_type == equipment_type and e.status == 'Available']
    
    def get_total_capacity(self) -> Dict[str, int]:
        """Get total equipment capacity by type"""
        capacity = defaultdict(int)
        for equip in self.equipment:
            if equip.status == 'Available':
                capacity[equip.equipment_type] += int(equip.speed * 10)  # Base capacity = 10
        return dict(capacity)

In [ ]:
class PredictiveAnalytics:
    """Predictive analytics for demand forecasting and optimization"""
    
    def __init__(self):
        self.demand_history = deque(maxlen=1000)
        self.weather_history = deque(maxlen=500)
        self.equipment_history = deque(maxlen=500)
        
    def update_history(self, timestamp: float, demand: int, weather: WeatherCondition, 
                      equipment_capacity: Dict[str, int]):
        """Update historical data"""
        self.demand_history.append((timestamp, demand))
        self.weather_history.append((timestamp, weather))
        self.equipment_history.append((timestamp, equipment_capacity))
    
    def forecast_demand(self, current_time: float, horizon: float) -> Dict[str, float]:
        """Forecast container demand for given horizon"""
        if len(self.demand_history) < 10:
            # Insufficient history - use simple trend
            recent_demand = [d for _, d in self.demand_history] if self.demand_history else [50]
            avg_demand = np.mean(recent_demand)
            return {
                'predicted_demand': avg_demand,
                'confidence': 0.5,
                'trend': 0.0
            }
        
        # Simple linear regression for trend
        recent_data = list(self.demand_history)[-20:]  # Last 20 data points
        times = [t for t, _ in recent_data]
        demands = [d for _, d in recent_data]
        
        # Calculate trend
        if len(times) > 1:
            trend = np.polyfit(times, demands, 1)[0]  # Slope
            predicted_demand = demands[-1] + trend * horizon
        else:
            trend = 0.0
            predicted_demand = demands[-1] if demands else 50
        
        # Calculate confidence based on variance
        variance = np.var(demands) if len(demands) > 1 else 0
        confidence = max(0.1, 1.0 - variance / (np.mean(demands) ** 2 + 1))
        
        return {
            'predicted_demand': max(0, predicted_demand),
            'confidence': confidence,
            'trend': trend
        }
    
    def predict_weather_impact(self, current_time: float) -> Dict[str, float]:
        """Predict weather impact on operations"""
        if len(self.weather_history) < 5:
            return {'impact_factor': 1.0, 'probability': 0.0}
        
        # Check recent weather patterns
        recent_weather = list(self.weather_history)[-10:]
        rain_events = [w for _, w in recent_weather if w.condition == 'Rain']
        
        if len(rain_events) > 0:
            rain_probability = len(rain_events) / len(recent_weather)
            avg_impact = np.mean([w.impact_factor for w in rain_events])
        else:
            rain_probability = 0.1  # Base 10% chance
            avg_impact = 0.8
        
        return {
            'impact_factor': avg_impact,
            'probability': rain_probability
        }
    
    def optimize_resource_allocation(self, predicted_demand: float, 
                                   equipment_capacity: Dict[str, int],
                                   weather_impact: float) -> Dict[str, Any]:
        """Optimize resource allocation based on predictions"""
        # Calculate required capacity
        required_yard_crane_capacity = predicted_demand / 10  # 10 containers per crane per hour
        required_quay_crane_capacity = predicted_demand / 15  # 15 containers per crane per hour
        
        # Apply weather impact
        required_yard_crane_capacity /= weather_impact
        required_quay_crane_capacity /= weather_impact
        
        # Compare with available capacity
        yard_utilization = required_yard_crane_capacity / max(1, equipment_capacity.get('YardCrane', 1))
        quay_utilization = required_quay_crane_capacity / max(1, equipment_capacity.get('QuayCrane', 1))
        
        return {
            'yard_utilization': min(1.0, yard_utilization),
            'quay_utilization': min(1.0, quay_utilization),
            'resource_adequacy': yard_utilization <= 1.0 and quay_utilization <= 1.0,
            'recommendations': self._generate_recommendations(yard_utilization, quay_utilization)
        }
    
    def _generate_recommendations(self, yard_util: float, quay_util: float) -> List[str]:
        """Generate optimization recommendations"""
        recommendations = []
        
        if yard_util > 0.9:
            recommendations.append("High yard crane utilization - consider additional equipment")
        elif yard_util < 0.5:
            recommendations.append("Low yard crane utilization - optimize scheduling")
        
        if quay_util > 0.9:
            recommendations.append("High quay crane utilization - check for bottlenecks")
        elif quay_util < 0.5:
            recommendations.append("Low quay crane utilization - review vessel schedule")
        
        if yard_util > 0.8 and quay_util > 0.8:
            recommendations.append("System-wide congestion detected - stagger operations")
        
        return recommendations

In [ ]:
class DigitalTwin:
    """Main Digital Twin system integrating all models"""
    
    def __init__(self, stacks: int, tiers: int, berths: int):
        self.stacks = stacks
        self.tiers = tiers
        self.berths = berths
        
        # Initialize models
        self.pre_marshalling = None
        self.vessel_schedule = None
        self.equipment_model = None
        self.predictive_analytics = PredictiveAnalytics()
        
        # Event system
        self.event_queue = deque()
        self.event_history = []
        
        # Simulation state
        self.current_time = 0.0
        self.simulation_speed = 1.0  # Hours per second
        
        # Performance metrics
        self.kpi_history = defaultdict(list)
        
    def initialize_scenario(self, containers: List[Container], 
                          vessels: List[VesselSchedule],
                          equipment: List[EquipmentStatus]):
        """Initialize digital twin with scenario data"""
        self.pre_marshalling = PreMarshallingModel(self.stacks, self.tiers, containers)
        self.vessel_schedule = VesselScheduleModel(vessels, self.berths)
        self.equipment_model = EquipmentModel(equipment)
        
        # Create initial event
        initial_event = DigitalTwinEvent(
            timestamp=0.0,
            event_type='simulation_start',
            source_system='digital_twin',
            data={'containers': len(containers), 'vessels': len(vessels), 'equipment': len(equipment)}
        )
        self.event_queue.append(initial_event)
    
    def add_event(self, event: DigitalTwinEvent):
        """Add event to the queue"""
        self.event_queue.append(event)
    
    def process_events(self) -> List[DigitalTwinEvent]:
        """Process all events in the queue"""
        processed_events = []
        
        while self.event_queue:
            event = self.event_queue.popleft()
            
            # Route event to appropriate model
            if event.source_system == 'vessel_schedule':
                self.vessel_schedule.update_schedule(self.current_time, [event])
            elif event.source_system == 'equipment':
                pass  # Equipment events are handled in main simulation loop
            elif event.source_system == 'weather':
                pass  # Weather events are handled in main simulation loop
            
            self.event_history.append(event)
            processed_events.append(event)
        
        return processed_events
    
    def simulate_step(self, dt: float = 0.1) -> Dict[str, Any]:
        """Simulate one time step"""
        self.current_time += dt
        
        # Generate weather conditions
        weather = self.generate_weather_condition()
        
        # Update models
        self.equipment_model.update_equipment_status(self.current_time, weather)
        
        # Process events
        processed_events = self.process_events()
        
        # Get current system state
        system_state = self.get_system_state()
        
        # Update predictive analytics
        equipment_capacity = self.equipment_model.get_total_capacity()
        self.predictive_analytics.update_history(
            self.current_time, 
            system_state['pending_containers'],
            weather,
            equipment_capacity
        )
        
        # Calculate KPIs
        kpis = self.calculate_kpis(system_state, weather, equipment_capacity)
        
        # Store KPIs
        for kpi, value in kpis.items():
            self.kpi_history[kpi].append((self.current_time, value))
        
        return {
            'timestamp': self.current_time,
            'system_state': system_state,
            'weather': weather,
            'equipment_capacity': equipment_capacity,
            'kpis': kpis,
            'events_processed': len(processed_events)
        }
    
    def generate_weather_condition(self) -> WeatherCondition:
        """Generate current weather condition"""
        # Simple weather model - 15% chance of rain
        if random.random() < 0.15:
            return WeatherCondition(
                timestamp=self.current_time,
                condition='Rain',
                impact_factor=0.8,
                duration=random.uniform(1, 4)
            )
        else:
            return WeatherCondition(
                timestamp=self.current_time,
                condition='Clear',
                impact_factor=1.0,
                duration=0.0
            )
    
    def get_system_state(self) -> Dict[str, Any]:
        """Get current system state"""
        urgent_containers = [c for c in self.pre_marshalling.containers 
                            if c.departure_time <= self.current_time + 4]  # Next 4 hours
        
        return {
            'total_containers': len(self.pre_marshalling.containers),
            'pending_containers': len(urgent_containers),
            'violations': self.pre_marshalling.count_violations(),
            'active_vessels': len([v for v in self.vessel_schedule.vessels 
                               if v.arrival_time <= self.current_time <= v.departure_time]),
            'available_yard_cranes': len(self.equipment_model.get_available_equipment('YardCrane')),
            'available_quay_cranes': len(self.equipment_model.get_available_equipment('QuayCrane'))
        }
    
    def calculate_kpis(self, system_state: Dict[str, Any], weather: WeatherCondition, 
                      equipment_capacity: Dict[str, int]) -> Dict[str, float]:
        """Calculate system KPIs"""
        # Efficiency KPIs
        loading_efficiency = self.vessel_schedule.get_loading_efficiency(self.current_time)
        equipment_utilization = sum(equipment_capacity.values()) / (len(equipment_capacity) * 10)  # Base capacity = 10
        
        # Predictive KPIs
        demand_forecast = self.predictive_analytics.forecast_demand(self.current_time, 4.0)  # 4-hour forecast
        weather_impact = self.predictive_analytics.predict_weather_impact(self.current_time)
        
        # Optimization recommendations
        resource_optimization = self.predictive_analytics.optimize_resource_allocation(
            demand_forecast['predicted_demand'],
            equipment_capacity,
            weather_impact['impact_factor']
        )
        
        return {
            'loading_efficiency': loading_efficiency,
            'equipment_utilization': equipment_utilization,
            'predicted_demand': demand_forecast['predicted_demand'],
            'demand_confidence': demand_forecast['confidence'],
            'weather_impact': weather_impact['impact_factor'],
            'yard_utilization': resource_optimization['yard_utilization'],
            'quay_utilization': resource_optimization['quay_utilization'],
            'resource_adequacy': 1.0 if resource_optimization['resource_adequacy'] else 0.0,
            'system_health': (loading_efficiency + equipment_utilization) / 2
        }

In [ ]:
try:
    def run_simulation_scenario() -> Dict[str, Any]:
        """Run complete digital twin simulation scenario"""
        print("=" * 60)
        print("DIGITAL TWIN SIMULATION - PRE-MARSHALLING OPTIMIZATION")
        print("=" * 60)
        
        # Initialize digital twin
        stacks, tiers, berths = 12, 6, 4
        digital_twin = DigitalTwin(stacks, tiers, berths)
        
        print(f"\nDigital Twin Configuration:")
        print(f"- Stacks: {stacks}")
        print(f"- Tiers: {tiers}")
        print(f"- Berths: {berths}")
        
        # Generate containers
        containers = []
        for i in range(2400):  # 2,400 TEU
            stack = random.randint(0, stacks - 1)
            tier = random.randint(0, tiers - 1)
            priority = random.randint(1, 100)
            vessel_id = f"V{random.randint(1, 3)}"
            departure_time = random.uniform(0, 18)  # 18-hour window
            
            containers.append(Container(
                id=f"C{i+1}",
                priority=priority,
                current_stack=stack,
                current_tier=tier,
                destination_vessel=vessel_id,
                departure_time=departure_time
            ))
        
        # Generate vessel schedules
        vessels = [
            VesselSchedule("V1", 2.0, 8.0, 800, 0),
            VesselSchedule("V2", 5.0, 12.0, 900, 1),
            VesselSchedule("V3", 10.0, 18.0, 700, 2)
        ]
        
        # Generate equipment
        equipment = []
        for i in range(4):  # 4 yard cranes
            equipment.append(EquipmentStatus(
                equipment_id=f"YC{i+1}",
                equipment_type="YardCrane",
                location=(random.randint(0, stacks-1), 0),
                status="Available"
            ))
        
        for i in range(3):  # 3 quay cranes
            equipment.append(EquipmentStatus(
                equipment_id=f"QC{i+1}",
                equipment_type="QuayCrane",
                location=(i, 0),
                status="Available"
            ))
        
        print(f"\nScenario Configuration:")
        print(f"- Containers: {len(containers)} TEU")
        print(f"- Vessels: {len(vessels)} (Total capacity: {sum(v.capacity for v in vessels)} TEU)")
        print(f"- Yard Cranes: {len([e for e in equipment if e.equipment_type == 'YardCrane'])}")
        print(f"- Quay Cranes: {len([e for e in equipment if e.equipment_type == 'QuayCrane'])}")
        
        # Initialize scenario
        digital_twin.initialize_scenario(containers, vessels, equipment)
        
        # Run simulation
        print(f"\nRunning 18-hour simulation...")
        
        simulation_steps = int(18.0 / 0.1)  # 0.1 hour steps
        results = {
            'timestamps': [],
            'loading_efficiency': [],
            'equipment_utilization': [],
            'predicted_demand': [],
            'weather_impact': [],
            'system_health': [],
            'violations': []
            'resource_adequacy': []
            'events': []
        }
        
        start_time = time.time()
        
        for step in range(simulation_steps):
            step_result = digital_twin.simulate_step(0.1)
            
            # Store results
            results['timestamps'].append(step_result['timestamp'])
            results['loading_efficiency'].append(step_result['kpis']['loading_efficiency'])
            results['equipment_utilization'].append(step_result['kpis']['equipment_utilization'])
            results['predicted_demand'].append(step_result['kpis']['predicted_demand'])
            results['weather_impact'].append(step_result['kpis']['weather_impact'])
            results['system_health'].append(step_result['kpis']['system_health'])
            results['violations'].append(step_result['system_state']['violations'])
            results['resource_adequacy'].append(step_result['kpis']['resource_adequacy'])
            results['events'].append(step_result['events_processed'])
            
            # Add random disruption events
            if random.random() < 0.01:  # 1% chance per step
                disruption_event = DigitalTwinEvent(
                    timestamp=step_result['timestamp'],
                    event_type=random.choice(['equipment_breakdown', 'vessel_delay', 'urgent_container']),
                    source_system='disruption',
                    data={'severity': random.uniform(0.5, 1.0)}
                )
                digital_twin.add_event(disruption_event)
            
            # Print progress
            if (step + 1) % 36 == 0:  # Every 3.6 hours
                current_health = step_result['kpis']['system_health']
                print(f"  Time: {step_result['timestamp']:.1f}h, System Health: {current_health:.2%}")
        
        simulation_time = time.time() - start_time
        
        print(f"\nSimulation completed in {simulation_time:.2f} seconds")
        print(f"Total events processed: {sum(results['events'])}")
        
        return results, digital_twin
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: invalid syntax. Perhaps you forgot a comma? (<string>, line 79)...]

In [ ]:
try:
    # Run the simulation
    simulation_results, digital_twin = run_simulation_scenario()
    
    # Analyze results
    print("\n" + "=" * 60)
    print("SIMULATION RESULTS ANALYSIS")
    print("=" * 60)
    
    # Calculate summary statistics
    final_health = simulation_results['system_health'][-1]
    avg_efficiency = np.mean(simulation_results['loading_efficiency'])
    avg_utilization = np.mean(simulation_results['equipment_utilization'])
    total_violations = sum(simulation_results['violations'])
    resource_adequacy_rate = np.mean(simulation_results['resource_adequacy'])
    
    print(f"\nSystem Performance Summary:")
    print(f"  Final system health: {final_health:.2%}")
    print(f"  Average loading efficiency: {avg_efficiency:.2%}")
    print(f"  Average equipment utilization: {avg_utilization:.2%}")
    print(f"  Total violations resolved: {total_violations}")
    print(f"  Resource adequacy rate: {resource_adequacy_rate:.2%}")
    
    # Performance classification
    if final_health > 0.8:
        performance_grade = "Excellent"
    elif final_health > 0.6:
        performance_grade = "Good"
    elif final_health > 0.4:
        performance_grade = "Fair"
    else:
        performance_grade = "Poor"
    
    print(f"  Overall performance: {performance_grade}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'run_simulation_scenario' is not defined...]

In [ ]:
try:
    # Create comprehensive visualization
    def visualize_digital_twin_results(results: Dict[str, Any]):
        """Visualize digital twin simulation results"""
        fig, axes = plt.subplots(3, 3, figsize=(18, 12))
        fig.suptitle('Digital Twin Pre-Marshalling Simulation Results', fontsize=16, fontweight='bold')
        
        # Convert to numpy arrays for easier handling
        timestamps = np.array(results['timestamps'])
        
        # 1. System Health Over Time
        axes[0, 0].plot(timestamps, results['system_health'], 'b-', linewidth=2)
        axes[0, 0].axhline(y=0.8, color='g', linestyle='--', alpha=0.7, label='Target (80%)')
        axes[0, 0].set_xlabel('Time (hours)')
        axes[0, 0].set_ylabel('System Health')
        axes[0, 0].set_title('System Health Over Time')
        axes[0, 0].grid(True, alpha=0.3)
        axes[0, 0].legend()
        
        # 2. Loading Efficiency
        axes[0, 1].plot(timestamps, results['loading_efficiency'], 'g-', linewidth=2)
        axes[0, 1].fill_between(timestamps, results['loading_efficiency'], alpha=0.3, color='green')
        axes[0, 1].set_xlabel('Time (hours)')
        axes[0, 1].set_ylabel('Loading Efficiency')
        axes[0, 1].set_title('Vessel Loading Efficiency')
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. Equipment Utilization
        axes[0, 2].plot(timestamps, results['equipment_utilization'], 'r-', linewidth=2)
        axes[0, 2].axhline(y=0.85, color='orange', linestyle='--', alpha=0.7, label='Optimal (85%)')
        axes[0, 2].set_xlabel('Time (hours)')
        axes[0, 2].set_ylabel('Equipment Utilization')
        axes[0, 2].set_title('Equipment Utilization')
        axes[0, 2].grid(True, alpha=0.3)
        axes[0, 2].legend()
        
        # 4. Predicted Demand
        axes[1, 0].plot(timestamps, results['predicted_demand'], 'purple', linewidth=2)
        axes[1, 0].set_xlabel('Time (hours)')
        axes[1, 0].set_ylabel('Predicted Demand')
        axes[1, 0].set_title('Demand Forecast (4-hour horizon)')
        axes[1, 0].grid(True, alpha=0.3)
        
        # 5. Weather Impact
        axes[1, 1].plot(timestamps, results['weather_impact'], 'cyan', linewidth=2)
        axes[1, 1].fill_between(timestamps, 1.0, results['weather_impact'], alpha=0.3, color='cyan')
        axes[1, 1].set_xlabel('Time (hours)')
        axes[1, 1].set_ylabel('Weather Impact Factor')
        axes[1, 1].set_title('Weather Impact on Operations')
        axes[1, 1].grid(True, alpha=0.3)
        
        # 6. Priority Violations
        axes[1, 2].plot(timestamps, results['violations'], 'orange', linewidth=2)
        axes[1, 2].set_xlabel('Time (hours)')
        axes[1, 2].set_ylabel('Priority Violations')
        axes[1, 2].set_title('Priority Violations Over Time')
        axes[1, 2].grid(True, alpha=0.3)
        
        # 7. Resource Adequacy
        axes[2, 0].plot(timestamps, results['resource_adequacy'], 'brown', linewidth=2)
        axes[2, 0].fill_between(timestamps, 0, results['resource_adequacy'], alpha=0.3, color='brown')
        axes[2, 0].set_xlabel('Time (hours)')
        axes[2, 0].set_ylabel('Resource Adequacy')
        axes[2, 0].set_title('Resource Adequacy (1=Adequate, 0=Inadequate)')
        axes[2, 0].grid(True, alpha=0.3)
        
        # 8. Event Activity
        axes[2, 1].plot(timestamps, np.cumsum(results['events']), 'red', linewidth=2)
        axes[2, 1].set_xlabel('Time (hours)')
        axes[2, 1].set_ylabel('Cumulative Events')
        axes[2, 1].set_title('Event Processing Activity')
        axes[2, 1].grid(True, alpha=0.3)
        
        # 9. Performance Radar Chart
        axes[2, 2].remove()
        ax9 = fig.add_subplot(3, 3, 9, projection='polar')
        
        # Calculate final metrics
        metrics = [
            results['system_health'][-1],
            np.mean(results['loading_efficiency']),
            np.mean(results['equipment_utilization']),
            resource_adequacy_rate,
            1.0 - (total_violations / len(results['violations'])),  # Inverse of violation rate
            np.mean(results['weather_impact'])  # Weather resilience
        ]
        
        labels = ['Health', 'Efficiency', 'Utilization', 'Resources', 'Compliance', 'Resilience']
        angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
        metrics += metrics[:1]  # Close the loop
        angles += angles[:1]
        
        ax9.plot(angles, metrics, 'b-', linewidth=2)
        ax9.fill(angles, metrics, alpha=0.25, color='blue')
        ax9.set_xticks(angles[:-1])
        ax9.set_xticklabels(labels)
        ax9.set_ylim(0, 1)
        ax9.set_title('System Performance Radar')
        ax9.grid(True)
        
        plt.tight_layout()
        plt.show()
    
    # Visualize results
    visualize_digital_twin_results(simulation_results)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'simulation_results' is not defined...]

In [ ]:
try:
    # What-if scenario analysis
    print("\n" + "=" * 60)
    print("WHAT-IF SCENARIO ANALYSIS")
    print("=" * 60)
    
    def run_what_if_scenarios() -> Dict[str, Dict]:
        """Run different what-if scenarios"""
        scenarios = {
            'baseline': {'equipment_failure': False, 'weather_severity': 1.0, 'demand_surge': 1.0},
            'equipment_failure': {'equipment_failure': True, 'weather_severity': 1.0, 'demand_surge': 1.0},
            'severe_weather': {'equipment_failure': False, 'weather_severity': 0.6, 'demand_surge': 1.0},
            'demand_surge': {'equipment_failure': False, 'weather_severity': 1.0, 'demand_surge': 1.5},
            'multiple_disruptions': {'equipment_failure': True, 'weather_severity': 0.7, 'demand_surge': 1.3}
        }
        
        results = {}
        
        for scenario_name, params in scenarios.items():
            print(f"\nRunning scenario: {scenario_name}")
            
            # Create modified digital twin for scenario
            stacks, tiers, berths = 12, 6, 4
            scenario_dt = DigitalTwin(stacks, tiers, berths)
            
            # Generate scenario-specific containers
            base_containers = 2400
            container_count = int(base_containers * params['demand_surge'])
            
            containers = []
            for i in range(container_count):
                stack = random.randint(0, stacks - 1)
                tier = random.randint(0, tiers - 1)
                priority = random.randint(1, 100)
                vessel_id = f"V{random.randint(1, 3)}"
                departure_time = random.uniform(0, 18)
                
                containers.append(Container(
                    id=f"C{i+1}",
                    priority=priority,
                    current_stack=stack,
                    current_tier=tier,
                    destination_vessel=vessel_id,
                    departure_time=departure_time
                ))
            
            # Standard vessels
            vessels = [
                VesselSchedule("V1", 2.0, 8.0, 800, 0),
                VesselSchedule("V2", 5.0, 12.0, 900, 1),
                VesselSchedule("V3", 10.0, 18.0, 700, 2)
            ]
            
            # Equipment with scenario modifications
            equipment = []
            for i in range(4):  # 4 yard cranes
                status = 'Breakdown' if params['equipment_failure'] and i == 0 else 'Available'
                equipment.append(EquipmentStatus(
                    equipment_id=f"YC{i+1}",
                    equipment_type="YardCrane",
                    location=(random.randint(0, stacks-1), 0),
                    status=status
                ))
            
            for i in range(3):  # 3 quay cranes
                equipment.append(EquipmentStatus(
                    equipment_id=f"QC{i+1}",
                    equipment_type="QuayCrane",
                    location=(i, 0),
                    status="Available"
                ))
            
            # Initialize scenario
            scenario_dt.initialize_scenario(containers, vessels, equipment)
            
            # Override weather generation for severe weather scenarios
            original_generate_weather = scenario_dt.generate_weather_condition
            
            def modified_weather():
                if params['weather_severity'] < 1.0:
                    return WeatherCondition(
                        timestamp=scenario_dt.current_time,
                        condition='Rain',
                        impact_factor=params['weather_severity'],
                        duration=random.uniform(2, 6)
                    )
                else:
                    return original_generate_weather()
            
            if params['weather_severity'] < 1.0:
                scenario_dt.generate_weather_condition = modified_weather
            
            # Run shorter simulation for what-if analysis
            scenario_results = {'system_health': [], 'loading_efficiency': [], 'violations': []}
            
            for step in range(90):  # 9 hours instead of 18
                step_result = scenario_dt.simulate_step(0.1)
                scenario_results['system_health'].append(step_result['kpis']['system_health'])
                scenario_results['loading_efficiency'].append(step_result['kpis']['loading_efficiency'])
                scenario_results['violations'].append(step_result['system_state']['violations'])
            
            # Calculate scenario metrics
            results[scenario_name] = {
                'avg_system_health': np.mean(scenario_results['system_health']),
                'final_system_health': scenario_results['system_health'][-1],
                'avg_loading_efficiency': np.mean(scenario_results['loading_efficiency']),
                'total_violations': sum(scenario_results['violations']),
                'resilience_score': scenario_results['system_health'][-1] / max(1, sum(scenario_results['violations']) / 100)
            }
            
            print(f"  Average system health: {results[scenario_name]['avg_system_health']:.2%}")
            print(f"  Final system health: {results[scenario_name]['final_system_health']:.2%}")
            print(f"  Loading efficiency: {results[scenario_name]['avg_loading_efficiency']:.2%}")
            print(f"  Total violations: {results[scenario_name]['total_violations']}")
        
        return results
    
    # Run what-if scenarios
    scenario_results = run_what_if_scenarios()
    
    # Visualize scenario comparison
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('What-If Scenario Analysis', fontsize=16, fontweight='bold')
    
    scenarios = list(scenario_results.keys())
    metrics = ['avg_system_health', 'final_system_health', 'avg_loading_efficiency', 'resilience_score']
    titles = ['Average System Health', 'Final System Health', 'Loading Efficiency', 'Resilience Score']
    axes = [ax1, ax2, ax3, ax4]
    
    for ax, metric, title in zip(axes, metrics, titles):
        values = [scenario_results[s][metric] for s in scenarios]
        bars = ax.bar(scenarios, values, alpha=0.7)
        
        # Color bars based on performance
        for bar, value in zip(bars, values):
            if metric == 'total_violations':
                bar.set_color('red' if value > 100 else 'orange' if value > 50 else 'green')
            else:
                bar.set_color('green' if value > 0.8 else 'orange' if value > 0.6 else 'red')
        
        ax.set_title(title)
        ax.set_ylabel('Score' if 'Score' in title else 'Value')
        ax.grid(True, alpha=0.3)
        
        # Rotate x labels for better readability
        ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Print scenario analysis summary
    print(f"\nScenario Analysis Summary:")
    print(f"{'Scenario':<20} {'Health':<10} {'Efficiency':<12} {'Violations':<12} {'Resilience':<12}")
    print("-" * 70)
    for scenario in scenarios:
        result = scenario_results[scenario]
        print(f"{scenario:<20} {result['final_system_health']:<10.2%} {result['avg_loading_efficiency']:<12.2%} "
              f"{result['total_violations']:<12} {result['resilience_score']:<12.3f}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: 0.1...]

### Why This Tier Exists vs Deep Reinforcement Learning

**Digital Twin Advantages:**
- **System-of-Systems Integration**: Connects multiple operational models for holistic optimization
- **Predictive Analytics**: Forecast future conditions and proactively optimize operations
- **Real-Time Synchronization**: Maintains virtual replica aligned with physical operations
- **What-If Analysis**: Test scenarios and strategies without risking actual operations
- **Cross-Functional Optimization**: Optimizes across vessel schedules, equipment, and yard operations

**Limitations of Deep Reinforcement Learning (Tier 4):**
- **Single Problem Focus**: Solves pre-marshalling in isolation without system context
- **No Predictive Capability**: Reactive rather than proactive decision making
- **Limited Integration**: Cannot account for external factors like weather or equipment failures
- **Static Environment**: Assumes fixed operational parameters

### Pros and Cons vs Alternative Approaches

**Pros:**
- ✓ Comprehensive system-wide optimization
- ✓ Predictive and proactive decision making
- ✓ Real-time adaptation to changing conditions
- ✓ Risk-free scenario testing and planning
- ✓ Integration of multiple operational models
- ✓ Resilience to disruptions and unexpected events
- ✓ Data-driven insights and recommendations

**Cons:**
- ✗ High computational complexity and resource requirements
- ✗ Complex implementation and maintenance
- ✗ Requires extensive data integration and synchronization
- ✗ Model accuracy dependent on quality of input data
- ✗ Significant upfront investment in infrastructure
- ✗ Requires specialized expertise to develop and operate

### When to Use This Tier

**Use Digital Twin when:**
- Complex, interconnected operations require optimization
- Predictive and proactive decision making is valuable
- Multiple external factors impact operations (weather, equipment, schedules)
- Risk-free scenario testing and planning is needed
- Real-time operational adjustments are critical
- System-wide performance optimization is the goal
- Sufficient resources exist for implementation and maintenance

**Consider alternatives when:**
- Simple, isolated optimization problems exist
- Limited computational resources are available
- Real-time integration is not required
- Problem domain is well-understood and stable
- Quick implementation is needed
- Budget constraints limit complex solutions
- Single-objective optimization is sufficient